# Data Preprocessing

The objective of this notebook is to transform the feature-engineered fraud dataset into a machine-learning-ready format.

The preprocessing pipeline includes:
- Feature selection
- Encoding categorical variables
- Scaling numerical variables
- Train-test splitting
- Handling class imbalance using SMOTE

In [43]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

import joblib

In [44]:
df = pd.read_csv(
    "../data/processed/fraud_geolocation.csv"
)

df.shape

(129146, 14)

In [45]:
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,lower_bound_ip_address,upper_bound_ip_address,country
0,247547,2015-06-28 03:00:34,2015-08-09 03:57:29,47,KIXYSVCHIPQBR,SEO,Safari,F,30,16778864,0,16778240.0,16779263.0,Australia
1,220737,2015-01-28 14:21:11,2015-02-11 20:28:28,15,PKYOWQKWGJNJI,SEO,Chrome,F,34,16842045,0,16809984.0,16842751.0,Thailand
2,390400,2015-03-19 20:49:09,2015-04-11 23:41:23,44,LVCSXLISZHVUO,Ads,IE,M,29,16843656,0,16843264.0,16843775.0,China
3,69592,2015-02-24 06:11:57,2015-05-23 16:40:14,55,UHAUHNXXUADJE,Direct,Chrome,F,30,16938732,0,16924672.0,16941055.0,China
4,174987,2015-07-07 12:58:11,2015-11-03 04:04:30,51,XPGPMOHIDRMGE,SEO,Chrome,F,37,16971984,0,16941056.0,16973823.0,Thailand


In [46]:
drop_cols = [
    "user_id",
    "device_id",
    "signup_time",
    "purchase_time"
]

df = df.drop(
    columns=drop_cols,
    errors="ignore"
)

In [47]:
y = df["class"]

X = df.drop(
    columns=["class"]
)

In [48]:
X.select_dtypes(
    include=["object"]
).columns

C:\Users\pc\AppData\Local\Temp\ipykernel_3916\185125415.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(


Index(['source', 'browser', 'sex', 'country'], dtype='str')

In [49]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [50]:
X.shape

(129146, 192)

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [52]:
print(X_train.shape)
print(X_test.shape)

(103316, 192)
(25830, 192)


In [53]:
print(y_train.value_counts())

class
0    93502
1     9814
Name: count, dtype: int64


In [54]:
print(
    y_train.value_counts(normalize=True) * 100
)

class
0    90.500987
1     9.499013
Name: proportion, dtype: float64


The training data remains highly imbalanced, with fraudulent transactions representing a very small percentage of observations.

In [55]:
numeric_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

In [56]:
scaler = StandardScaler()

In [57]:
X_train[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)

In [58]:
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

['../models/scaler.pkl']

In [1]:
smote = SMOTE(
    random_state=42
)

X_train_resampled, y_train_resampled = (
    smote.fit_resample(
        X_train,
        y_train
    )
)

NameError: name 'SMOTE' is not defined

In [60]:
print(
    y_train_resampled.value_counts()
)

class
0    93502
1    93502
Name: count, dtype: int64


In [61]:
print(
    y_train_resampled.value_counts(normalize=True) * 100
)

class
0    50.0
1    50.0
Name: proportion, dtype: float64


In [62]:
X_train_resampled.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

pd.Series(
    y_train_resampled
).to_csv(
    "../data/processed/y_train.csv",
    index=False
)

In [63]:
X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

pd.Series(
    y_test
).to_csv(
    "../data/processed/y_test.csv",
    index=False
)

## Preprocessing Summary

The preprocessing pipeline transformed the raw fraud transaction data into a machine-learning-ready dataset. Categorical variables were encoded using one-hot encoding, numerical features were standardized using StandardScaler, and class imbalance was addressed using SMOTE on the training set only. The resulting datasets are prepared for model development and evaluation.